In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:
monthly = pd.read_csv("data/monthly.csv")

short_names = {
    'retail_and_recreation_percent_change_from_baseline': 'Retail/Recreation',
    'grocery_and_pharmacy_percent_change_from_baseline': 'Grocery/Pharmacy',
    'transit_stations_percent_change_from_baseline': 'Transit Stations',
    'workplaces_percent_change_from_baseline': 'Workplaces',
    'residential_percent_change_from_baseline': 'Residential'
}

monthly = monthly.rename(columns=short_names)

mobility_feats_short = ['Retail/Recreation', 'Grocery/Pharmacy', 'Transit Stations', 'Workplaces', 'Residential']

In [ ]:
# linearity: scatterplots
import itertools

pairs = list(itertools.combinations(mobility_feats_short, 2))

sns.set_style("darkgrid")
fig, axes = plt.subplots(4, 3, figsize=(15, 20))
axes = axes.flatten()

for ax, (x, y) in zip(axes, pairs):
    sns.scatterplot(data=monthly, x=x, y=y, ax=ax)
    ax.set_xlabel(x, style='italic')
    ax.set_ylabel(y, style='italic')
    ax.set_title(f"{x} vs {y}")

x_last, y_last = pairs[-1]

sns.scatterplot(data=monthly, x=x_last, y=y_last, ax=axes[10])
axes[10].set_title(f"{x_last} vs {y_last}")
axes[10].set_xlabel(x_last, style='italic')
axes[10].set_ylabel(y_last, style='italic')

# hide unused axes: 9 and 11
axes[9].set_visible(False)
axes[11].set_visible(False)

plt.tight_layout()
plt.show()



In [ ]:
# correlation
mob_monthly = monthly[mobility_feats_short]

mob_corr = mob_monthly.corr()
mask = np.tril(np.ones_like(mob_corr, dtype=bool), k=-1)

sns.set_style("dark")
plt.figure(figsize=(8, 6))
sns.heatmap(mob_corr, annot=True, mask=mask)
plt.title("Correlation Matrix of Mobility Categories")
plt.show()


In [ ]:
# outliers

sns.set_style("darkgrid")
fig, axes = plt.subplots(2, 3, figsize=(16, 12))

axes = axes.flatten()

for ax, var in zip(axes, mobility_feats_short): 
    sns.boxplot(data=monthly, y=var, ax=ax)
    ax.set_ylabel(var, style='italic')
    ax.set_title(var)

var_last = mobility_feats_short[-1]
log_res = np.log10(monthly["Residential"])
inv_res = 1/monthly['Residential']
sqr_res = monthly['Residential']**2
sqrt_res = monthly['Residential']**0.5 # best transformation

sns.boxplot(data=monthly, y=var_last, ax=axes[5])
axes[5].set_ylabel(var_last, style='italic')
axes[5].set_title(var)

# hide unused axes
axes[4].set_visible(False)

plt.tight_layout()
plt.show()

In [ ]:
# outliers

sns.set_style("darkgrid")
fig, axes = plt.subplots(2, 3, figsize=(16, 12))

axes = axes.flatten()

for ax, var in zip(axes, mobility_feats_short): 
    sns.histplot(data=monthly, x=var, ax=ax)
    ax.set_xlabel(var, style='italic')
    ax.set_title(var)

var_last = mobility_feats_short[-1]
log_res = np.log10(monthly["Residential"])
inv_res = 1/monthly['Residential']
sqr_res = monthly['Residential']**2
sqrt_res = monthly['Residential']**0.5 # best transformation

sns.histplot(data=monthly, x=var_last, ax=axes[5])
axes[5].set_xlabel(var_last, style='italic')
axes[5].set_title(var)

# hide unused axes
axes[4].set_visible(False)

plt.tight_layout()
plt.show()

In [ ]:
print(mob_monthly.size)

In [ ]:
# outlier flagging: 

rr_mad = np.median(abs(monthly['Retail/Recreation']-np.median(monthly['Retail/Recreation'])))
gp_mad = np.median(abs(monthly['Grocery/Pharmacy']-np.median(monthly['Grocery/Pharmacy'])))
t_mad = np.median(abs(monthly['Transit Stations']-np.median(monthly['Transit Stations'])))
w_mad = np.median(abs(monthly['Workplaces']-np.median(monthly['Workplaces'])))
res_mad = np.median(abs(monthly['Residential']-np.median(monthly['Residential'])))


mobility_feats_norm = ["Retail/Recreation", "Grocery/Pharmacy"]
mobility_feats_skew = ["Transit Stations", "Workplaces", "Residential"]


# mad z for normal

def count_out_mad_z(feat, df=None): 
    if df is not None: 
        x = df[feat]
    else: 
        x = feat
    med = np.median(df[feat])
    kmad = 1.4826*np.median(abs(df[feat]-med))

    out = (abs((x-med)/kmad) > 3.5).sum()

    print(f"\n{feat}: {out}")


for feat in mobility_feats_norm: 
    count_out_mad_z(feat, monthly)

# left/right for skewed
def count_out_mad_skew(feat, df=None): 
    if df is not None: 
        x = df[feat]

    else: 
        x = pd.Series(feat)

    med = np.median(x)
    left = x[x <= med]
    right = x[x >= med]

    mad_l = np.median(abs(left-med))
    mad_r = np.median(abs(right-med))

    lb = med-(3.5*mad_l)
    ub = med+(3.5*mad_r)
    
    out = ((x<lb) | (x>ub)).sum()

    print(f"\n{feat}: {out}")
    

for feat in mobility_feats_skew: 
    count_out_mad_skew(feat, monthly)



# print(f"\n\n0.5% of data: {monthly.size*0.005:.2f}")

# for feat in mobility_feats_short: 
#     perc_low = np.percentile(monthly[feat], 0.005)
#     perc_up = np.percentile(monthly[feat], 0.995)
#     out = ((monthly[feat] < perc_low) | (monthly[feat] > perc_up)).sum()
#     print(f"\n{feat}: {out}, {out/monthly.size*100:.2f}%")

